## Python REPL Tool

In [ ]:
# pip install langchain-openai langchain-experimental langchain


In [ ]:
from langchain_experimental.tools import PythonREPLTool
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

import os
from dotenv import load_dotenv

In [ ]:
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('Voer je OpenAI API key in: ')

In [ ]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')



In [ ]:
os.environ['NO_PROXY'] = '*'

llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)

In [ ]:
base_repl = PythonREPLTool()

@tool
def safe_python_repl(code: str) -> str:
    """
    Use this tool to write and execute Python code.
    Use it for calculations, simulations, data analysis, or anything that requires running code.
    Input should be valid Python code as a string.
    """
    print(f"\n--- CODE TO BE EXECUTED ---\n{code}\n---------------------------")
    approval = interrupt("Keur je deze code goed? (yes/no): ")
    if approval.strip().lower() == "yes":
        return base_repl.run(code)
    return "Execution rejected by user."

In [ ]:
from IPython.display import Markdown, display

checkpointer = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[safe_python_repl],
    checkpointer=checkpointer,
)

task = """
Simulate the motion of a ball dropped from 100 meters height under gravity (g = 9.81 m/s^2).
Write a Python function that computes height after time t.
Then calculate the height at t=0, 1, 2, 3, 4, and 5 seconds.
Additionally provide the code used
"""

config = {"configurable": {"thread_id": "1"}}

# Start de agent
agent.invoke({"messages": [{"role": "user", "content": task}]}, config=config)

# Behandel interrupts in een loop
while True:
    state = agent.get_state(config)
    if not state.next:
        break  # klaar, geen interrupts meer

    # Toon de vraag van de interrupt
    vraag = state.tasks[0].interrupts[0].value
    antwoord = input(f"\n{vraag} ")

    # Hervat de agent met het antwoord
    agent.invoke(Command(resume=antwoord), config=config)

# Eindresultaat
state = agent.get_state(config)
display(Markdown(state.values["messages"][-1].content))